# 01 · Arquitectura del Data Lake

Objetivo: reconocer las capas, sus productos y la separación de ámbitos. El código comprueba qué tablas existen y visualiza su relación.

**Requisitos.** Python, las dependencias fijadas y el ZIP del proyecto. En Colab se solicita el ZIP si el código no está disponible y la celda de preparación instala los paquetes faltantes. En local se usa un entorno virtual.

**Pasos.** Leer el diagrama, ejecutar una corrida y consultar el catálogo DuckDB. [Arquitectura](../docs/arquitectura.md) · [Diagrama editable](../figures/flujo.md).

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import os
import subprocess
import sys
import tempfile
import zipfile

package_name = "SLP-BigData-Reproducible"
here = Path.cwd().resolve()
explicit = os.getenv("SLP_BIGDATA_ROOT")
candidates = [Path(explicit).expanduser().resolve()] if explicit else [
    here, here.parent, here / package_name,
]
ROOT = next((p for p in candidates if (p / "src" / "repro.py").is_file()), None)
colab = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if ROOT is None and colab:
    from google.colab import files
    print("Selecciona el ZIP SLP-BigData-Reproducible.zip.")
    uploaded = files.upload()
    if set(uploaded) != {package_name + ".zip"}:
        raise ValueError("Selecciona únicamente el ZIP de distribución del proyecto.")
    with zipfile.ZipFile(package_name + ".zip") as archive:
        for entry in archive.infolist():
            parts = Path(entry.filename).parts
            if not parts or parts[0] != package_name or ".." in parts or "\\" in entry.filename:
                raise ValueError("El ZIP contiene una ruta ajena al paquete.")
        archive.extractall(here)
    ROOT = here / package_name
if ROOT is None or not (ROOT / "src" / "repro.py").is_file():
    raise FileNotFoundError("Abre la carpeta descomprimida o define SLP_BIGDATA_ROOT.")

requirements = ROOT / "requirements.txt"
def current_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None
pins = [line.split("==", 1) for line in requirements.read_text(encoding="utf-8").splitlines()
        if "==" in line and not line.startswith("#")]
missing = [(name, version) for name, version in pins if current_version(name) != version]
if missing:
    if not colab:
        raise RuntimeError("Instala requirements.txt en el entorno virtual activo antes de continuar.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
sys.path.insert(0, str(ROOT))
from src.repro import run_demo, run_original, run_benchmark, query, read_report

private = ROOT / ".bigdata-private"
private.mkdir(exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix="notebook-", dir=private))
print("Dependencias verificadas:", {name: current_version(name) for name, _ in pins})

In [ ]:
# A ejecuta una demostración sintética. B verifica archivos autorizados.
MODO = "A"
INAH = private / "input-inah"
DATATUR = private / "input-datatur"

def ejecutar(modo, salida):
    if modo == "A":
        return run_demo(salida)
    if modo == "B":
        if not INAH.is_dir() or not DATATUR.is_dir():
            print("Modo B requiere los originales autorizados. Consulta docs/reproducibilidad.md.")
            return None
        return run_original(INAH, DATATUR, salida)
    raise ValueError("MODO admite A o B.")

In [ ]:
print("Raw → Bronze → Silver → Gold → DuckDB → SQL")
print("INAH: recinto × mes | DataTur: centro hotelero × mes")
producto = ejecutar(MODO, WORK / "arquitectura")
if producto is not None:
    db = producto / "warehouse" / "campaign.duckdb"
    for fila in query(db, "SHOW TABLES"):
        print(fila)

**Resultado esperado.** Aparecen las dos tablas de hechos y el calendario. La demostración usa entidades ficticias. El periodo alinea los resultados, cada hecho mantiene su unidad. [Diccionario](../docs/diccionario_datos.md).